In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
import matplotlib.pyplot as plt
import os
import pathlib
import numpy as np

# --- Configuration ---
DATA_DIR = pathlib.Path('images')
BATCH_SIZE = 64
IMG_HEIGHT = 96
IMG_WIDTH = 96
EPOCHS = 50
NUM_CLASSES_TO_USE = 101

print(f"TensorFlow version: {tf.__version__}")
print(f"Num GPUs Available: {len(tf.config.list_physical_devices('GPU'))}")

# --- Data Loading ---
print(f"Selecting top {NUM_CLASSES_TO_USE} classes...")
all_classes = sorted([d.name for d in DATA_DIR.iterdir() if d.is_dir()])
selected_classes = all_classes[:NUM_CLASSES_TO_USE]
print(f"Selected classes: {selected_classes[:5]}...")

# Collect file paths and labels
file_paths = []
labels = []
class_to_idx = {cls: i for i, cls in enumerate(selected_classes)}

for cls in selected_classes:
    cls_dir = DATA_DIR / cls
    for img_path in cls_dir.glob("*.jpg"):
        file_paths.append(str(img_path))
        labels.append(class_to_idx[cls])

print(f"Total images: {len(file_paths)}")

file_paths = np.array(file_paths)
labels = np.array(labels)

np.random.seed(123)
indices = np.random.permutation(len(file_paths))
file_paths = file_paths[indices]
labels = labels[indices]

split_idx = int(len(file_paths) * 0.8)
train_paths = file_paths[:split_idx]
train_labels = labels[:split_idx]
val_paths = file_paths[split_idx:]
val_labels = labels[split_idx:]

print(f"\nTraining set: {len(train_paths)} images")
print(f"Validation set: {len(val_paths)} images")

# Create Dataset
def process_path(file_path, label):
    img = tf.io.read_file(file_path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, [IMG_HEIGHT, IMG_WIDTH])
    return img, label

def augment(img, label):
    img = tf.image.random_flip_left_right(img)
    return img, label

# Train dataset
train_ds = tf.data.Dataset.from_tensor_slices((train_paths.tolist(), train_labels.tolist()))
train_ds = train_ds.shuffle(buffer_size=5000, seed=123)
train_ds = train_ds.map(process_path, num_parallel_calls=tf.data.AUTOTUNE)
train_ds = train_ds.map(augment, num_parallel_calls=tf.data.AUTOTUNE)
train_ds = train_ds.batch(BATCH_SIZE).prefetch(buffer_size=2)

# Validation dataset
val_ds = tf.data.Dataset.from_tensor_slices((val_paths.tolist(), val_labels.tolist()))
val_ds = val_ds.map(process_path, num_parallel_calls=tf.data.AUTOTUNE)
val_ds = val_ds.batch(BATCH_SIZE).prefetch(buffer_size=2)

# --- ResNet50 AVEC RÉGULARISATION ---
print("\n Construction de ResNet50 from scratch...")

# Ajouter un regularizer pour stabiliser
base_model = ResNet50(
    weights=None,
    include_top=False,
    input_shape=(IMG_HEIGHT, IMG_WIDTH, 3)
)

base_model.trainable = True

# Modèle avec régularisation
model = models.Sequential([
    layers.Rescaling(1./255, input_shape=(IMG_HEIGHT, IMG_WIDTH, 3)),
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.BatchNormalization(), 
    layers.Dense(256, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(0.01)), 
    layers.Dropout(0.5),
    layers.Dense(NUM_CLASSES_TO_USE, activation='softmax')
])

print(" ResNet50 from scratch construit!")
model.summary()

# --- Compilation avec LEARNING RATE ---
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),  
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False),
    metrics=['accuracy']
)

# --- Callbacks ---
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True,
    verbose=1
)

checkpoint = ModelCheckpoint(
    'best_resnet_scratch.h5',
    monitor='val_accuracy',
    save_best_only=True,
    mode='max',
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=4,
    min_lr=1e-7,
    verbose=1
)

# --- Training ---
print("\n Entraînement ResNet50 from scratch ...\n")

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=[early_stopping, checkpoint, reduce_lr]
)

# --- Visualization ---
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
loss = history.history['loss']
val_loss = history.history['val_loss']

epochs_range = range(len(acc))

plt.figure(figsize=(12, 6))

plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, label='Training Accuracy')
plt.plot(epochs_range, val_acc, label='Validation Accuracy')
plt.legend(loc='lower right')
plt.title('Training and Validation Accuracy')

plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, label='Training Loss')
plt.plot(epochs_range, val_loss, label='Validation Loss')
plt.legend(loc='upper right')
plt.title('Training and Validation Loss')

plt.show()

model.save('food_resnet_scratch_final.h5')
print("\n Model saved!")

TensorFlow version: 2.16.2
Num GPUs Available: 1
Selecting top 101 classes...
Selected classes: ['apple_pie', 'baby_back_ribs', 'baklava', 'beef_carpaccio', 'beef_tartare']...
Total images: 101000

Training set: 80800 images
Validation set: 20200 images

 Construction de ResNet50 from scratch...
 ResNet50 from scratch construit!


Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ rescaling_4 (Rescaling)         │ (None, 96, 96, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ resnet50 (Functional)           │ (None, 3, 3, 2048)     │    23,587,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_4      │ (None, 2048)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 2048)           │         8,192 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 256)            │       524,544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_7 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_12 (Dense)                │ (None, 101)            │        25,957 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 24,146,405 (92.11 MB)

 Trainable params: 24,089,189 (91.89 MB)

...


Epoch 13: finished saving model to best_resnet_scratch.h5
1263/1263 ━━━━━━━━━━━━━━━━━━━━ 28166s 22s/step - accuracy: 0.3296 - loss: 2.9390 - val_accuracy: 0.2701 - val_loss: 4.2163 - learning_rate: 1.0000e-04
Epoch 14/50
1263/1263 ━━━━━━━━━━━━━━━━━━━━ 0s 522ms/step - accuracy: 0.3535 - loss: 2.7954
Epoch 14: val_accuracy improved from 0.27010 to 0.30307, saving model to best_resnet_scratch.h5



Epoch 14: finished saving model to best_resnet_scratch.h5
1263/1263 ━━━━━━━━━━━━━━━━━━━━ 693s 549ms/step - accuracy: 0.3639 - loss: 2.7438 - val_accuracy: 0.3031 - val_loss: 2.9938 - learning_rate: 1.0000e-04
Epoch 15/50
1263/1263 ━━━━━━━━━━━━━━━━━━━━ 0s 522ms/step - accuracy: 0.3910 - loss: 2.5314
Epoch 15: val_accuracy did not improve from 0.30307
1263/1263 ━━━━━━━━━━━━━━━━━━━━ 694s 549ms/step - accuracy: 0.4027 - loss: 2.4807 - val_accuracy: 0.2999 - val_loss: 3.0618 - learning_rate: 1.0000e-04
Epoch 16/50
1263/1263 ━━━━━━━━━━━━━━━━━━━━ 0s 537ms/step - accuracy: 0.4291 - loss: 2.3384
Epoch 16: val_accuracy improved from 0.30307 to 0.30901, saving model to best_resnet_scratch.h5



Epoch 16: finished saving model to best_resnet_scratch.h5
1263/1263 ━━━━━━━━━━━━━━━━━━━━ 719s 569ms/step - accuracy: 0.4405 - loss: 2.2995 - val_accuracy: 0.3090 - val_loss: 3.0421 - learning_rate: 1.0000e-04
Epoch 17/50
1263/1263 ━━━━━━━━━━━━━━━━━━━━ 0s 603ms/step - accuracy: 0.4717 - loss: 2.1634
Epoch 17: val_accuracy did not improve from 0.30901
1263/1263 ━━━━━━━━━━━━━━━━━━━━ 804s 636ms/step - accuracy: 0.4802 - loss: 2.1294 - val_accuracy: 0.3063 - val_loss: 3.1377 - learning_rate: 1.0000e-04
Epoch 18/50
1263/1263 ━━━━━━━━━━━━━━━━━━━━ 0s 624ms/step - accuracy: 0.5050 - loss: 2.0174
Epoch 18: val_accuracy improved from 0.30901 to 0.32149, saving model to best_resnet_scratch.h5



Epoch 18: finished saving model to best_resnet_scratch.h5

Epoch 18: ReduceLROnPlateau reducing learning rate to 4.999999873689376e-05.
1263/1263 ━━━━━━━━━━━━━━━━━━━━ 830s 656ms/step - accuracy: 0.5160 - loss: 1.9707 - val_accuracy: 0.3215 - val_loss: 3.0454 - learning_rate: 1.0000e-04
Epoch 19/50
1263/1263 ━━━━━━━━━━━━━━━━━━━━ 0s 641ms/step - accuracy: 0.5868 - loss: 1.6750
Epoch 19: val_accuracy improved from 0.32149 to 0.35074, saving model to best_resnet_scratch.h5



Epoch 19: finished saving model to best_resnet_scratch.h5
1263/1263 ━━━━━━━━━━━━━━━━━━━━ 849s 672ms/step - accuracy: 0.6150 - loss: 1.5610 - val_accuracy: 0.3507 - val_loss: 2.9817 - learning_rate: 5.0000e-05
Epoch 20/50
1263/1263 ━━━━━━━━━━━━━━━━━━━━ 0s 583ms/step - accuracy: 0.6496 - loss: 1.4191
Epoch 20: val_accuracy did not improve from 0.35074
1263/1263 ━━━━━━━━━━━━━━━━━━━━ 772s 611ms/step - accuracy: 0.6673 - loss: 1.3440 - val_accuracy: 0.3403 - val_loss: 3.1334 - learning_rate: 5.0000e-05
Epoch 21/50
1263/1263 ━━━━━━━━━━━━━━━━━━━━ 0s 541ms/step - accuracy: 0.6915 - loss: 1.2444
Epoch 21: val_accuracy did not improve from 0.35074
1263/1263 ━━━━━━━━━━━━━━━━━━━━ 719s 569ms/step - accuracy: 0.7093 - loss: 1.1749 - val_accuracy: 0.3503 - val_loss: 3.1886 - learning_rate: 5.0000e-05
Epoch 22/50
1263/1263 ━━━━━━━━━━━━━━━━━━━━ 0s 554ms/step - accuracy: 0.7322 - loss: 1.0734
Epoch 22: val_accuracy did not improve from 0.35074
1263/1263 ━━━━━━━━━━━━━━━━━━━━ 738s 584ms/step - accuracy: 